# Zoektermen groeperen op wat Google werkelijk toont

Dit notebook haalt voor elke zoekterm de eerste zoekresultaten van Google op en gebruikt
die om drie vragen te beantwoorden.

1. **Welke zoektermen horen bij elkaar?** Zoektermen die dezelfde resultaten opleveren,
   beantwoordt Google met hetzelfde soort pagina. Die horen op één pagina.
2. **Welke onderdelen toont Google?** Een AI-overzicht, andere vragen, een videoblok,
   een kaart. Dat bepaalt hoeveel ruimte er voor een gewoon resultaat overblijft.
3. **Wie staat er het vaakst?** Welke domeinen nemen in jouw verzameling zoektermen de
   meeste ruimte in.

Aan het eind krijg je een Excel-werkblad met je zoektermen gelabeld, een PDF-rapport
om te lezen, en drie CSV-bestanden.

**Wat je nodig hebt**

- Een lijst zoektermen. Hoe je die verzamelt staat op de uitlegpagina
- Een account bij DataForSEO met wat tegoed erop. Twintig zoektermen kosten ongeveer zeven dollarcent
- Ongeveer tien minuten

**Waar je op moet letten.** Het ophalen kost geld per zoekterm. Het notebook rekent
vooraf uit wat een run kost en stopt als dat boven je eigen grens uitkomt.

---

## Stap 1 — de benodigdheden installeren

Klik op het driehoekje links van het blok hieronder. Dit duurt ongeveer twintig seconden
en hoeft maar één keer per sessie.

In [ ]:
!pip install --quiet fpdf2 openpyxl requests matplotlib
print("Klaar. Ga door naar stap 2.")

---

## Stap 2 — je instellingen

Vul je inloggegevens van DataForSEO in en pas zo nodig de rest aan. Dit stel je
één keer in. Je zoektermen komen in de volgende stap, in een blok apart.

Een paar dingen om te weten.

- **`DIEPTE`** is hoeveel resultaten er per zoekterm worden opgehaald. Twintig is genoeg.
- **`TOP_N`** is hoeveel van die resultaten meetellen bij het vergelijken. Tien is de
  gangbare keuze, want dat is ongeveer wat iemand op de eerste pagina ziet.
- **`SAMEN_VANAF`** is het aantal gedeelde resultaten waarbij twee zoektermen als
  hetzelfde onderwerp gelden. Drie van de tien is een goed startpunt. Zet je hem hoger,
  dan krijg je kleinere en zuiverdere groepen.
- **`VERWANT_VANAF`** is de lagere drempel. Zoektermen die deze halen maar niet de
  hoge, horen bij elkaar maar verdienen een eigen pagina. Die krijgen hetzelfde
  familienummer.
- **`MAX_KOSTEN`** is je plafond. Het notebook stopt voordat het geld uitgeeft als de
  raming daarboven uitkomt.

In [ ]:
# ── Inloggegevens DataForSEO ───────────────────────────────────────────────────
# Vervang deze twee regels door je eigen gegevens. Deel dit notebook niet
# opnieuw met je gegevens erin.
DATAFORSEO_LOGIN    = "jouw@email.nl"
DATAFORSEO_WACHTWOORD = "jouw-api-wachtwoord"

# ── Waar en hoe je meet ────────────────────────────────────────────────────────
LOCATIE_CODE = 2528          # 2528 = Nederland, 2056 = België, 2840 = Verenigde Staten
LOCATIE_NAAM = "Nederland"   # alleen voor op het rapport
TAAL         = "nl"
APPARAAT     = "desktop"     # of "mobile"
DIEPTE       = 20            # hoeveel resultaten ophalen per zoekterm
TOP_N        = 10            # hoeveel daarvan meetellen bij het vergelijken

# ── De drempels ────────────────────────────────────────────────────────────────
SAMEN_VANAF   = 3            # gedeelde resultaten: zelfde onderwerp
VERWANT_VANAF = 2            # gedeelde resultaten: zelfde familie

# ── Grenzen en eigen domein ────────────────────────────────────────────────────
MAX_KOSTEN    = 1.00         # in dollar. Hierboven stopt het notebook
EIGEN_DOMEIN  = ""           # bijvoorbeeld "jouwbedrijf.nl". Leeg mag ook

print("Instellingen klaar. Ga door naar stap 3.")

---

## Stap 3 — je zoektermen

Er zijn twee manieren. Kies er één, je hoeft niets aan de code te veranderen.

**Plakken.** Zet je zoektermen in het blok hieronder, één per regel, tussen de drie
aanhalingstekens. Geen komma's, geen aanhalingstekens per regel. Lege regels en
regels die met een `#` beginnen worden overgeslagen.

**Een bestand kiezen.** Zet `UPLOADEN` op `True` en draai de cel. Er verschijnt een
knop waarmee je een CSV- of Excel-bestand kiest, bijvoorbeeld een export uit Search
Console. Het notebook zoekt zelf de kolom met zoektermen; vindt het die niet, dan
noemt het de kolommen die er wel zijn zodat je er één kunt aanwijzen.

Dubbele zoektermen worden er automatisch uitgehaald, want elke zoekterm kost geld.
Werkt vanaf ongeveer tien zoektermen. Boven de tweehonderd wordt het traag en duur.

In [ ]:
# ── Plakken ────────────────────────────────────────────────────────────────────
# Eén zoekterm per regel. Vervang de voorbeelden hieronder door je eigen lijst.
ZOEKTERMEN_TEKST = """
seo rapportage
seo rapport maken
seo rapportage voorbeeld
seo rapportage tool
marketing dashboard
marketing dashboard maken
marketing dashboard voorbeeld
looker studio dashboard
zoekwoorden clusteren
zoektermen groeperen
keyword clustering
topical authority
"""

# ── Of een bestand kiezen ──────────────────────────────────────────────────────
UPLOADEN = False   # True = een CSV of Excel kiezen in plaats van het blok hierboven
KOLOM    = ""      # leeg laten: het notebook zoekt zelf. Anders de kolomnaam invullen


# Vanaf hier hoef je niets aan te passen. ──────────────────────────────────────
import csv, io, re

KOPNAMEN = ["zoekterm", "zoektermen", "zoekopdracht", "zoekopdrachten",
            "belangrijkste zoekopdrachten", "query", "queries", "top queries",
            "keyword", "keywords", "term", "termen", "trefwoord", "trefwoorden"]


def _rijen_uit_bestand(naam, inhoud):
    """Leest een CSV of Excel en geeft de rijen terug als lijsten met tekst."""
    if naam.lower().endswith((".xlsx", ".xlsm")):
        from openpyxl import load_workbook
        blad = load_workbook(io.BytesIO(inhoud), read_only=True, data_only=True).active
        return [["" if c is None else str(c) for c in rij]
                for rij in blad.iter_rows(values_only=True)]

    for codering in ("utf-8-sig", "utf-16", "latin-1"):
        try:
            tekst = inhoud.decode(codering)
            break
        except UnicodeDecodeError:
            continue
    # Search Console levert komma's, Excel in Nederland vaak puntkomma's.
    scheiding = ";" if tekst.count(";") > tekst.count(",") else ","
    return [rij for rij in csv.reader(io.StringIO(tekst), delimiter=scheiding)]


def _kolom_kiezen(kop):
    """Welke kolom bevat de zoektermen? Eerst de wens, dan de bekende namen."""
    schoon = [k.strip().lower() for k in kop]
    if KOLOM:
        if KOLOM.strip().lower() in schoon:
            return schoon.index(KOLOM.strip().lower())
        raise SystemExit(f"De kolom '{KOLOM}' staat niet in het bestand. "
                         f"Gevonden kolommen: {', '.join(kop)}")
    for i, naam in enumerate(schoon):
        if naam in KOPNAMEN:
            return i
    if len(kop) == 1:
        return 0
    raise SystemExit("Ik kan niet zien welke kolom de zoektermen bevat. Vul KOLOM "
                     f"hierboven in met een van deze namen: {', '.join(kop)}")


def _uit_bestand():
    from google.colab import files
    print("Kies je CSV- of Excel-bestand.")
    gekozen = files.upload()
    if not gekozen:
        raise SystemExit("Er is geen bestand gekozen.")
    naam = list(gekozen)[0]
    rijen = [r for r in _rijen_uit_bestand(naam, gekozen[naam]) if any(v.strip() for v in r)]
    if not rijen:
        raise SystemExit(f"{naam} is leeg.")

    kolom = _kolom_kiezen(rijen[0])
    kop_is_kop = rijen[0][kolom].strip().lower() in KOPNAMEN or bool(KOLOM)
    print(f"{naam}: kolom '{rijen[0][kolom].strip()}' gebruikt."
          if kop_is_kop else f"{naam}: eerste kolom gebruikt, geen kopregel gevonden.")
    return [r[kolom] for r in (rijen[1:] if kop_is_kop else rijen) if len(r) > kolom]


def _opschonen(ruw):
    """Witruimte weg, lege regels en # weg, dubbele weg, volgorde behouden."""
    gezien, uit, dubbel = set(), [], 0
    for regel in ruw:
        term = re.sub(r"\s+", " ", str(regel)).strip().strip('"').strip("'")
        if not term or term.startswith("#"):
            continue
        if term.lower() in gezien:
            dubbel += 1
            continue
        gezien.add(term.lower())
        uit.append(term)
    return uit, dubbel


ZOEKTERMEN, dubbel = _opschonen(
    _uit_bestand() if UPLOADEN else ZOEKTERMEN_TEKST.splitlines())

if not ZOEKTERMEN:
    raise SystemExit("Er staan geen zoektermen in. Vul het blok hierboven of kies "
                     "een bestand.")

print(f"\n{len(ZOEKTERMEN)} zoektermen klaar" + (f", {dubbel} dubbele overgeslagen" if dubbel else ""))
print("De eerste vijf: " + ", ".join(ZOEKTERMEN[:5]))
if len(ZOEKTERMEN) < 5:
    print("\nLet op: met minder dan vijf zoektermen valt er weinig te groeperen.")
if len(ZOEKTERMEN) > 200:
    print(f"\nLet op: {len(ZOEKTERMEN)} zoektermen duurt een paar minuten en kost meer. "
          "Kijk in stap 5 naar de raming voordat je doorgaat.")
print("\nGa door naar stap 4.")

---

## Stap 4 — de motor inladen

Hier hoef je niets aan te veranderen. Dit blok laadt het rekenwerk in. Klik op het
driehoekje en ga door naar stap 5.

In [ ]:
import base64, io, json, re, time, datetime
from collections import Counter, defaultdict
from pathlib import Path
import requests

API = "https://api.dataforseo.com/v3/serp/google/organic/live/advanced"

def haal_serps(zoektermen, login, wachtwoord, *, locatie=2528, taal="nl",
               diepte=20, apparaat="desktop", pauze=0.0, log=print):
    """Haalt per zoekterm de zoekresultaten op. Geeft {zoekterm: [resultaten]}.

    Eén aanroep per zoekterm, want DataForSEO rekent per zoekterm af en dan is
    de kostenteller eerlijk. Een mislukte zoekterm stopt de rest niet.
    """
    sessie = requests.Session()
    sessie.auth = (login, wachtwoord)
    uitkomst = {}
    kosten = 0.0
    mislukt = []

    for i, term in enumerate(zoektermen, 1):
        payload = [{
            "keyword": term,
            "location_code": locatie,
            "language_code": taal,
            "device": apparaat,
            "depth": diepte,
        }]
        try:
            antwoord = sessie.post(API, json=payload, timeout=90)
            data = antwoord.json()
        except Exception as fout:                      # netwerk of json
            mislukt.append((term, str(fout)))
            log(f"  [{i}/{len(zoektermen)}] {term} — mislukt ({fout})")
            continue

        if data.get("status_code") != 20000:
            mislukt.append((term, data.get("status_message", "onbekend")))
            log(f"  [{i}/{len(zoektermen)}] {term} — {data.get('status_message')}")
            continue

        kosten += data.get("cost", 0) or 0
        taken = data.get("tasks") or []
        items = []
        if taken and taken[0].get("result"):
            items = taken[0]["result"][0].get("items") or []
        uitkomst[term] = items
        log(f"  [{i}/{len(zoektermen)}] {term} — {len(items)} resultaten")
        if pauze:
            time.sleep(pauze)

    return uitkomst, round(kosten, 4), mislukt


# ---------------------------------------------------------------- opschonen
def domein(url):
    """Haalt het domein uit een adres, zonder www."""
    if not url:
        return ""
    m = re.match(r"https?://([^/]+)", url)
    if not m:
        return ""
    return m.group(1).lower().removeprefix("www.")


def organieke_urls(items, top_n):
    """De organische resultaten uit een SERP, in volgorde, tot top_n."""
    urls = []
    for item in items:
        if item.get("type") != "organic":
            continue
        url = item.get("url")
        if not url:
            continue
        urls.append(url.split("#")[0].rstrip("/").lower())
        if len(urls) >= top_n:
            break
    return urls


def onderdelen(items):
    """Welke onderdelen staan er in dit zoekresultaat, met hun hoogste positie.

    Geeft {soort: beste_positie}. De positie telt, want een videoblok onderaan
    doet iets anders dan een videoblok bovenaan.
    """
    gevonden = {}
    for item in items:
        soort = item.get("type")
        if not soort or soort == "organic":
            continue
        positie = item.get("rank_absolute") or item.get("rank_group") or 99
        if soort not in gevonden or positie < gevonden[soort]:
            gevonden[soort] = positie
    return gevonden


# ---------------------------------------------------------------- groeperen
def gedeeld(a, b):
    """Hoeveel van dezelfde adressen staan in beide zoekresultaten."""
    return len(set(a) & set(b))


def overlap(a, b):
    """Aandeel gedeelde adressen (0 tot 1). Naast het aantal, als context."""
    sa, sb = set(a), set(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def woorden(term):
    return set(re.findall(r"[a-z0-9]+", term.lower()))


def bevat_elkaar(a, b):
    """Bevat de ene zoekterm de andere, op woordniveau."""
    wa, wb = woorden(a), woorden(b)
    return wa < wb or wb < wa


def bouw_paren(serps, top_n):
    """Alle zoektermparen met het aantal gedeelde resultaten.

    Het aantal gedeelde adressen is de maat, niet een verhoudingsgetal. Dat is
    uit te leggen ("drie van de tien resultaten zijn hetzelfde") en het is wat
    er in de praktijk gebruikt wordt.
    """
    termen = sorted(serps)
    urls = {t: organieke_urls(serps[t], top_n) for t in termen}
    paren = []
    for i, a in enumerate(termen):
        for b in termen[i + 1:]:
            paren.append({
                "a": a,
                "b": b,
                "gedeeld": gedeeld(urls[a], urls[b]),
                "overlap": round(overlap(urls[a], urls[b]), 3),
                "bevat": bevat_elkaar(a, b),
            })
    return sorted(paren, key=lambda p: (-p["gedeeld"], -p["overlap"]))


def groepeer(serps, paren, min_gedeeld):
    """Groepeert zoektermen die minstens `min_gedeeld` resultaten delen.

    Verbonden groepen: hangt A aan B en B aan C, dan zitten ze in een groep,
    ook als A en C elkaar niet raken. Dat is bewust, want een onderwerp is een
    keten en geen kliek.
    """
    ouder = {t: t for t in serps}

    def wortel(x):
        while ouder[x] != x:
            ouder[x] = ouder[ouder[x]]
            x = ouder[x]
        return x

    for paar in paren:
        if paar["gedeeld"] >= min_gedeeld:
            ra, rb = wortel(paar["a"]), wortel(paar["b"])
            if ra != rb:
                ouder[rb] = ra

    groepen = defaultdict(list)
    for term in serps:
        groepen[wortel(term)].append(term)
    return sorted(groepen.values(), key=lambda g: (-len(g), g[0]))


def bouw_clusters(serps, top_n, hoog, laag):
    """Twee niveaus: onderwerpen (hoge drempel) binnen families (lage drempel).

    `hoog` en `laag` zijn aantallen gedeelde resultaten, niet percentages.
    """
    paren = bouw_paren(serps, top_n)
    onderwerpen = groepeer(serps, paren, hoog)
    families = groepeer(serps, paren, laag)

    familie_van = {}
    for i, familie in enumerate(families):
        for term in familie:
            familie_van[term] = i

    resultaat = []
    for onderwerp in onderwerpen:
        resultaat.append({
            "termen": sorted(onderwerp),
            "familie": familie_van[onderwerp[0]],
            "omvang": len(onderwerp),
        })
    # Op familie sorteren, zodat verwante onderwerpen onder elkaar staan.
    resultaat.sort(key=lambda c: (c["familie"], -c["omvang"], c["termen"][0]))
    return resultaat, families, paren


# ---------------------------------------------------------------- analyse

def tel_onderdelen(serps):
    """Bij hoeveel zoektermen komt elk onderdeel voor, en hoe prominent."""
    voorkomen = Counter()
    gewicht = defaultdict(float)
    for items in serps.values():
        for soort, positie in onderdelen(items).items():
            voorkomen[soort] += 1
            gewicht[soort] += 1.0 / max(positie, 1)
    return [
        {
            "onderdeel": soort,
            "zoektermen": aantal,
            "aandeel": round(aantal / max(len(serps), 1), 3),
            "prominentie": round(gewicht[soort], 2),
        }
        for soort, aantal in voorkomen.most_common()
    ]


def tel_concurrenten(serps, top_n, eigen_domein=""):
    """Welke domeinen komen het vaakst voor, gewogen naar positie."""
    voorkomen = Counter()
    gewicht = defaultdict(float)
    for items in serps.values():
        for positie, url in enumerate(organieke_urls(items, top_n), 1):
            d = domein(url)
            if not d:
                continue
            voorkomen[d] += 1
            gewicht[d] += 1.0 / positie
    rijen = [
        {
            "domein": d,
            "zoektermen": aantal,
            "aandeel": round(aantal / max(len(serps), 1), 3),
            "zichtbaarheid": round(gewicht[d], 2),
            "is_eigen": bool(eigen_domein) and d == eigen_domein.lower().removeprefix("www."),
        }
        for d, aantal in voorkomen.items()
    ]
    return sorted(rijen, key=lambda r: -r["zichtbaarheid"])


NL_NAMEN = {
    "organic": "Gewone resultaten",
    "paid": "Advertenties",
    "people_also_ask": "Andere vragen",
    "related_searches": "Gerelateerde zoekopdrachten",
    "video": "Video",
    "images": "Afbeeldingen",
    "local_pack": "Kaart met bedrijven",
    "map": "Kaart",
    "shopping": "Productblok",
    "featured_snippet": "Uitgelicht antwoord",
    "knowledge_graph": "Kennispaneel",
    "top_stories": "Nieuws",
    "twitter": "Social",
    "carousel": "Carrousel",
    "answer_box": "Antwoordblok",
    "ai_overview": "AI-overzicht",
    "discussions_and_forums": "Discussies en forums",
    "find_results_on": "Vind resultaten op",
    "jobs": "Vacatures",
    "recipes": "Recepten",
    "math_solver": "Rekenhulp",
    "google_flights": "Vluchten",
    "google_hotels": "Hotels",
    "events": "Evenementen",
    "podcasts": "Podcasts",
    "currency_box": "Valuta",
    "refine_products": "Productfilters",
    "popular_products": "Populaire producten",
    "commercial_units": "Productadvertenties",
    "short_videos": "Korte video's",
    "questions_and_answers": "Vragen en antwoorden",
    "visual_stories": "Verhalen",
    "explore_brands": "Merken verkennen",
    "third_party_reviews": "Beoordelingen",
    "app": "Apps",
    "scholarly_articles": "Wetenschappelijke artikelen",
}


def nl(soort):
    return NL_NAMEN.get(soort, soort.replace("_", " ").capitalize())

print("Motor geladen. Ga door naar stap 5.")

---

## Stap 5 — de zoekresultaten ophalen

**Dit is het blok dat geld kost.** Het rekent eerst uit wat de run ongeveer kost en
stopt als dat boven je `MAX_KOSTEN` uitkomt.

Reken op ongeveer een seconde per zoekterm. Bij twintig zoektermen ben je in een halve
minuut klaar; bij tweehonderd duurt het een paar minuten. Laat het tabblad open staan.

In [ ]:
PRIJS_PER_ZOEKTERM = 0.0035          # ruime schatting, de echte kosten worden geteld
raming = len(ZOEKTERMEN) * PRIJS_PER_ZOEKTERM

print(f"{len(ZOEKTERMEN)} zoektermen, geraamde kosten ongeveer ${raming:.2f}")

if raming > MAX_KOSTEN:
    raise SystemExit(
        f"Gestopt. De raming (${raming:.2f}) ligt boven je grens van ${MAX_KOSTEN:.2f}. "
        f"Verhoog MAX_KOSTEN in stap 2 of gebruik minder zoektermen."
    )

serps, kosten, mislukt = haal_serps(
    ZOEKTERMEN, DATAFORSEO_LOGIN, DATAFORSEO_WACHTWOORD,
    locatie=LOCATIE_CODE, taal=TAAL, diepte=DIEPTE, apparaat=APPARAAT,
)

print(f"\nOpgehaald: {len(serps)} van de {len(ZOEKTERMEN)} zoektermen")
print(f"Werkelijke kosten: ${kosten}")
if mislukt:
    print("\nNiet gelukt:")
    for term, reden in mislukt:
        print(f"  {term} — {reden}")

---

## Stap 6 — de zoektermen groeperen

Twee zoektermen horen bij hetzelfde onderwerp als ze genoeg van dezelfde resultaten
delen. Zoektermen die de lagere drempel halen krijgen hetzelfde **familienummer**: ze
horen bij elkaar, maar verdienen een eigen pagina.

Kijk vooral naar de gevallen die je niet had verwacht. Twee zoektermen die op elkaar
lijken en tóch uit elkaar vallen, zeggen dat Google er een andere vraag in leest.

In [ ]:
clusters, families, paren = bouw_clusters(serps, TOP_N, SAMEN_VANAF, VERWANT_VANAF)

print(f"{len(serps)} zoektermen  ->  {len(clusters)} onderwerpen in {len(families)} families\n")
print(f"{'FAM':<5}{'AANTAL':<8}ZOEKTERMEN")
print("-" * 78)
for c in clusters:
    print(f"{c['familie'] + 1:02d}   {c['omvang']:<8}{', '.join(c['termen'])}")

print("\nDe sterkst verbonden paren:")
for p in paren[:12]:
    if p["gedeeld"] == 0:
        break
    print(f"  {p['gedeeld']} van {TOP_N} gedeeld   {p['a']}  <->  {p['b']}")

---

## Stap 7 — wat Google in deze zoekresultaten toont

Twee overzichten. Welke onderdelen er in het zoekresultaat staan, en welke domeinen er
het vaakst in voorkomen.

De zichtbaarheid bij de domeinen is gewogen naar positie: een eerste plek telt zwaarder
dan een tiende. Zo zie je wie er werkelijk ruimte inneemt in plaats van wie er alleen
maar vaak ergens staat.

In [ ]:
onderdeel_rijen = tel_onderdelen(serps)
concurrent_rijen = tel_concurrenten(serps, TOP_N, eigen_domein=EIGEN_DOMEIN)

print("WAT GOOGLE TOONT")
print("-" * 60)
for r in onderdeel_rijen:
    if r["onderdeel"] == "organic":
        continue
    balk = "#" * int(r["aandeel"] * 40)
    print(f"  {r['aandeel'] * 100:5.0f}%  {balk:<40} {nl(r['onderdeel'])}")

print("\nWIE ER HET VAAKST STAAT")
print("-" * 60)
print(f"  {'ZICHTB.':<10}{'ZOEKT.':<10}DOMEIN")
for r in concurrent_rijen[:20]:
    merk = "   <- jij" if r["is_eigen"] else ""
    print(f"  {r['zichtbaarheid']:<10.2f}{r['zoektermen']:<10}{r['domein']}{merk}")

---

## Stap 8 — het rapport en de bestanden

Dit blok maakt drie dingen. Een **Excel-werkblad** met je zoektermen gelabeld, en dat is
het bestand waar je mee verder werkt. Een **PDF-rapport** om te lezen en door te sturen.
En drie **CSV-bestanden** voor wie liever zelf rekent. In Colab worden ze meteen gedownload.

Het werkblad heeft vijf tabbladen. Het eerste is de kern: elke zoekterm met zijn onderwerp,
zijn familie en in gewone taal wat dat betekent. De regels die oranje zijn, zijn de
onderwerpen waar meer dan één zoekterm in zit; daar valt de winst te halen.

De opmaak van het rapport zit in dit notebook zelf, er wordt geen sjabloon van een
website opgehaald. De twee lettertypes komen van Google Fonts; lukt dat niet, dan
gebruikt het rapport een standaardletter en gaat de rest gewoon door.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# Alle namen hieronder beginnen met XL_. In een notebook draaien alle cellen in
# dezelfde ruimte, en het rapport gebruikt dezelfde woorden (KAART, ACCENT,
# TEKST) voor kleuren in een ander formaat. Zonder voorvoegsel overschrijft de
# ene cel de andere en krijg je een foutmelding die nergens op slaat.
XL_DONKER = "FF0D1A22"
XL_KAART = "FF16272F"
XL_ACCENT = "FFFF5F1F"
XL_ACCENT_ZACHT = "FFFFE9DF"
XL_WIT = "FFFFFFFF"
XL_GRIJS = "FF5E6F78"
XL_LIJN = "FFE3E7E9"

XL_KOP = Font(name="Sora", size=10, bold=True, color=XL_WIT)
XL_TEKST = Font(name="Sora", size=10)
XL_MONO = Font(name="Consolas", size=9, color=XL_GRIJS)
XL_NADRUK = Font(name="Sora", size=10, bold=True)
XL_KOPJE = Font(name="Sora", size=11, bold=True, color="FF0D1A22")
XL_TITEL = Font(name="Sora", size=14, bold=True, color=XL_WIT)
XL_ONDERTITEL = Font(name="Sora", size=9, color="FFB9C4C9")

XL_VUL_DONKER = PatternFill("solid", fgColor=XL_DONKER)
XL_VUL_ACCENT = PatternFill("solid", fgColor=XL_ACCENT_ZACHT)
XL_RAND_ONDER = Border(bottom=Side(style="thin", color=XL_LIJN))


def _kopband(blad, titel, toelichting, breedte):
    """Twee regels donkere band bovenaan met de titel en een korte uitleg."""
    blad.merge_cells(start_row=1, start_column=1, end_row=1, end_column=breedte)
    blad.merge_cells(start_row=2, start_column=1, end_row=2, end_column=breedte)
    cel = blad.cell(row=1, column=1, value=titel)
    cel.font = XL_TITEL
    cel.alignment = Alignment(vertical="center", indent=1)
    cel2 = blad.cell(row=2, column=1, value=toelichting)
    cel2.font = XL_ONDERTITEL
    cel2.alignment = Alignment(vertical="center", indent=1)
    for rij in (1, 2):
        for kolom in range(1, breedte + 1):
            blad.cell(row=rij, column=kolom).fill = XL_VUL_DONKER
    blad.row_dimensions[1].height = 26
    blad.row_dimensions[2].height = 18
    blad.row_dimensions[3].height = 6


def _tabel(blad, kolommen, rijen, startrij, naam, breedtes):
    """Kolomkoppen plus rijen, met filter en bevroren koprij."""
    for i, kop in enumerate(kolommen, 1):
        cel = blad.cell(row=startrij, column=i, value=kop)
        cel.font = XL_KOP
        cel.fill = PatternFill("solid", fgColor=XL_KAART)
        cel.alignment = Alignment(vertical="center", indent=1)
    blad.row_dimensions[startrij].height = 20

    for r, rij in enumerate(rijen, startrij + 1):
        for k, waarde in enumerate(rij, 1):
            cel = blad.cell(row=r, column=k, value=waarde)
            cel.font = XL_TEKST
            cel.border = XL_RAND_ONDER
            cel.alignment = Alignment(vertical="center", indent=1, wrap_text=(k == len(rij)))

    for i, breedte in enumerate(breedtes, 1):
        blad.column_dimensions[get_column_letter(i)].width = breedte

    eind = startrij + len(rijen)
    if rijen:
        verwijzing = f"A{startrij}:{get_column_letter(len(kolommen))}{eind}"
        tabel = Table(displayName=naam, ref=verwijzing)
        tabel.tableStyleInfo = TableStyleInfo(name="TableStyleLight1", showRowStripes=False)
        blad.add_table(tabel)
    blad.freeze_panes = blad.cell(row=startrij + 1, column=1)
    return eind


def _leesmij(wb, *, zoektermen, top_n, samen_vanaf, verwant_vanaf, datum, locatie_naam):
    """Een tabblad dat het bestand uitlegt.

    Dit bestand wordt gedownload, later weer geopend en soms doorgestuurd naar
    een collega die er niet bij was. Eén regel boven elk tabblad is dan te
    weinig; de uitleg hoort in het bestand zelf te staan.
    """
    blad = wb.active
    blad.title = "Lees mij"
    _kopband(blad, "Zoektermanalyse",
             f"Gemaakt op {datum} · {len(zoektermen)} zoektermen · {locatie_naam}", 2)

    blokken = [
        ("Wat dit bestand is", [
            "Je zoektermen, gegroepeerd op de zoekresultaten die Google er werkelijk bij toont.",
            "Niet op woorden die op elkaar lijken, maar op hoeveel van dezelfde webadressen er "
            "bij twee zoektermen in het zoekresultaat staan.",
        ]),
        ("Onderwerp en familie, het verschil", [
            f"Twee zoektermen horen bij hetzelfde ONDERWERP als ze minstens {samen_vanaf} van de "
            f"{top_n} resultaten delen. Google beantwoordt ze dan met dezelfde pagina's, dus daar "
            "hoort één pagina te komen.",
            f"Halen ze dat niet maar delen ze er wel minstens {verwant_vanaf}, dan zitten ze in "
            "dezelfde FAMILIE. Verwant, maar elk een eigen pagina, met een link naar elkaar.",
        ]),
        ("Hoe je het eerste tabblad leest", [
            "De oranje regels zijn de onderwerpen waar meer dan één zoekterm in zit. Daar valt de "
            "winst te halen: die zoektermen horen samen op één pagina.",
            "Staat een zoekterm alleen, dan deelt hij weinig met de rest van je lijst. Dat kan een "
            "eigen onderwerp zijn, of een teken dat er zoektermen ontbreken die de brug vormen.",
            "Kijk vooral naar wat je niet had verwacht. Twee zoektermen die op elkaar lijken en "
            "tóch uit elkaar vallen, zeggen dat Google er een andere vraag in leest dan jij.",
        ]),
        ("De tabbladen", [
            "Zoektermen — de kern. Elke zoekterm met zijn onderwerp en wat dat betekent.",
            "Onderwerpen — dezelfde groepering, maar dan één regel per onderwerp.",
            "Wat Google toont — welke onderdelen er in deze zoekresultaten stonden.",
            "Domeinen — wie er het vaakst stond, gewogen naar positie.",
            "Gedeelde resultaten — de onderbouwing, per paar zoektermen.",
        ]),
        ("Waarmee dit gemeten is", [
            f"Zoektermen: {len(zoektermen)}",
            f"Land en taal: {locatie_naam}",
            f"Resultaten vergeleken: de eerste {top_n} per zoekterm",
            f"Zelfde onderwerp vanaf: {samen_vanaf} gedeelde resultaten",
            f"Zelfde familie vanaf: {verwant_vanaf} gedeelde resultaten",
            f"Gemeten op: {datum}",
        ]),
        ("Wat dit bestand niet doet", [
            "Het zegt niet welke zoekterm de belangrijkste is. Daar is zoekvolume of je eigen "
            "data voor nodig.",
            "Het kijkt niet naar je eigen site, alleen naar wat er in de zoekresultaten staat.",
            "Het schrijft geen content en geeft geen titeladvies.",
            "Een zoekresultaat is een momentopname. Over een paar maanden kan de groepering "
            "anders liggen.",
        ]),
        ("Waar het vandaan komt", [
            "Gemaakt met het zoektermscript van ferryjansen.nl/tools/zoektermen-groeperen",
            "Daar staat ook de uitleg over hoe je zoektermen verzamelt en wat het ophalen kost.",
        ]),
    ]

    rij = 4
    for titel, regels in blokken:
        cel = blad.cell(row=rij, column=1, value=titel)
        cel.font = XL_KOPJE
        cel.alignment = Alignment(vertical="center", indent=1)
        blad.row_dimensions[rij].height = 22
        rij += 1
        for regel in regels:
            cel = blad.cell(row=rij, column=1, value=regel)
            cel.font = XL_TEKST
            cel.alignment = Alignment(vertical="top", indent=1, wrap_text=True)
            blad.row_dimensions[rij].height = 15 * (1 + len(regel) // 95)
            rij += 1
        rij += 1

    blad.column_dimensions["A"].width = 108
    blad.column_dimensions["B"].width = 2
    blad.sheet_view.showGridLines = False
    return blad


def maak_werkblad(pad, *, clusters, onderdeel_rijen, concurrent_rijen, paren,
                  zoektermen, top_n, samen_vanaf, verwant_vanaf, datum, locatie_naam):
    wb = Workbook()

    _leesmij(wb, zoektermen=zoektermen, top_n=top_n, samen_vanaf=samen_vanaf,
             verwant_vanaf=verwant_vanaf, datum=datum, locatie_naam=locatie_naam)

    # ── De gelabelde zoektermenlijst ──────────────────────────────────────────
    # Dit is het blad waar iemand mee verder werkt. Eén regel per zoekterm, met
    # het label erbij en een advies in gewone taal.
    blad = wb.create_sheet("Zoektermen")
    _kopband(blad, "Je zoektermen, gelabeld",
             f"{len(zoektermen)} zoektermen · {locatie_naam} · gemeten op {datum} · "
             f"zelfde onderwerp vanaf {samen_vanaf} van de {top_n} gedeelde resultaten", 5)

    per_term = {}
    for nummer, cluster in enumerate(clusters, 1):
        for term in cluster["termen"]:
            per_term[term] = (nummer, cluster["familie"] + 1, cluster["omvang"])

    familie_omvang = {}
    for _, familie, _ in per_term.values():
        familie_omvang[familie] = familie_omvang.get(familie, 0) + 1

    rijen = []
    for term in sorted(zoektermen, key=lambda t: (per_term.get(t, (99, 99, 0))[1],
                                                  per_term.get(t, (99, 99, 0))[0], t)):
        nummer, familie, omvang = per_term.get(term, ("", "", 0))
        if omvang > 1:
            advies = "Samen op één pagina met de andere zoektermen in dit onderwerp"
        elif familie != "" and familie_omvang.get(familie, 0) > 1:
            advies = "Eigen pagina, met een link naar de andere onderwerpen in deze familie"
        else:
            advies = "Staat op zichzelf. Eigen pagina, of je lijst mist de tussenliggende zoektermen"
        rijen.append([term, familie, nummer, omvang, advies])

    eind = _tabel(blad, ["Zoekterm", "Familie", "Onderwerp", "Zoektermen in onderwerp", "Wat dit betekent"],
                  rijen, 4, "Zoektermen", [38, 10, 12, 22, 62])

    # Onderwerpen met meer dan één zoekterm oranje, want daar valt de winst
    for r in range(5, eind + 1):
        if (blad.cell(row=r, column=4).value or 0) > 1:
            for k in range(1, 6):
                blad.cell(row=r, column=k).fill = XL_VUL_ACCENT
            blad.cell(row=r, column=1).font = XL_NADRUK

    # ── Blad 2: de onderwerpen ────────────────────────────────────────────────
    blad = wb.create_sheet("Onderwerpen")
    _kopband(blad, "De onderwerpen",
             "Elk onderwerp is één pagina. Onderwerpen met hetzelfde familienummer "
             "horen bij elkaar en verdienen een link naar elkaar.", 4)
    _tabel(blad, ["Familie", "Onderwerp", "Aantal zoektermen", "Zoektermen"],
           [[c["familie"] + 1, i, c["omvang"], ", ".join(c["termen"])]
            for i, c in enumerate(clusters, 1)],
           4, "Onderwerpen", [10, 12, 20, 90])

    # ── Blad 3: wat Google toont ──────────────────────────────────────────────
    blad = wb.create_sheet("Wat Google toont")
    _kopband(blad, "Wat Google in deze zoekresultaten toont",
             "Bij welk deel van je zoektermen verschijnt welk onderdeel. Prominentie "
             "weegt mee naar positie: bovenaan telt zwaarder dan onderaan.", 4)
    _tabel(blad, ["Onderdeel", "Aantal zoektermen", "Aandeel", "Prominentie"],
           [[nl(r["onderdeel"]), r["zoektermen"], r["aandeel"], r["prominentie"]]
            for r in onderdeel_rijen if r["onderdeel"] != "organic"],
           4, "Onderdelen", [34, 20, 12, 14])
    for r in range(5, 5 + len(onderdeel_rijen)):
        blad.cell(row=r, column=3).number_format = "0%"

    # ── Blad 4: de domeinen ───────────────────────────────────────────────────
    blad = wb.create_sheet("Domeinen")
    _kopband(blad, "Wie er het vaakst staat",
             "Gewogen naar positie. Dit is wie er in jouw verzameling zoektermen de "
             "meeste ruimte inneemt, en dat is niet altijd je commerciële concurrent.", 4)
    _tabel(blad, ["Domein", "Aantal zoektermen", "Aandeel", "Zichtbaarheid"],
           [[r["domein"], r["zoektermen"], r["aandeel"], r["zichtbaarheid"]]
            for r in concurrent_rijen],
           4, "Domeinen", [40, 20, 12, 14])
    for r in range(5, 5 + len(concurrent_rijen)):
        blad.cell(row=r, column=3).number_format = "0%"

    # ── Blad 5: de paren, om zelf na te rekenen ───────────────────────────────
    blad = wb.create_sheet("Gedeelde resultaten")
    _kopband(blad, "Hoeveel resultaten elk paar zoektermen deelt",
             f"De onderbouwing van de groepering. Vanaf {samen_vanaf} gedeeld is het "
             f"hetzelfde onderwerp, vanaf {verwant_vanaf} dezelfde familie.", 4)
    _tabel(blad, ["Zoekterm A", "Zoekterm B", f"Gedeeld van {top_n}", "Verhouding"],
           [[p["a"], p["b"], p["gedeeld"], p["overlap"]]
            for p in paren if p["gedeeld"] > 0],
           4, "Paren", [38, 38, 16, 14])

    wb.save(pad)
    return pad

In [ ]:
from fpdf import FPDF

# Huisstijl
ACHTERGROND = (13, 26, 34)
KAART = (22, 39, 47)
RAND = (31, 52, 62)
TEKST = (244, 246, 247)
GEDEMPT = (138, 155, 163)
SUBTIEL = (94, 111, 120)
ACCENT = (255, 95, 31)

# Het logo staat als tekst in dit bestand, om dezelfde reden als de opmaak:
# het notebook moet zelfstandig werken en niets van een website hoeven halen.
LOGO_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAKAAAACgCAYAAACLz2ctAAAKhElEQVR42u2dfahkZR3HP7/nnLl3XySX1MTMl5S0QJbIjCJ3"
    "KRI3lkpYV7P+KKIyKIJqAyEQLAgJayGCytyyN8y0XKJ3egFbjJRNjBQ2ylxWLRL1tuu+3DvnnOfXH/Mc72m8uzsvZ+91dr4f"
    "OMyde2fmzpz5zO95nt/ze54xGrh7ZmaVu+fAVuA9wHrgdOAUhBicI8DTwB7gLuAHZnbI3YOZxfpG1i9fURSb8jz/PHBp3wO6"
    "zqkYAuu7vge4ycx+2JTQ+iLfJ4Ht6Q4RKIEcCDqfYkS6QJYOYoxfzrLsE+6eAdFq+cqy/GCWZTuSeDHdobb4WaBI1xUJxfEi"
    "nyd/XtoIXmW6zIHtZrbN3TMDWFhYuGRmZubP6Y/1nQFuBW4H/g7M69yKIZgBzgWuBj6VxhARqIBOWZZXdzqde+r+38+8R9fd"
    "K3d/riiKq3QORRu4++vc/bHkWOHu0d33uPtqc/dLgfv7Qud1qbPYaTTJQozSHGdmVrj7euCPwJpGF+9aqqr6XDJzIV3+Nlnb"
    "0fkTLUXATrrcnhybT1HwzhBCeG3f7X/k7hpsiDaJ7h6KotjZGIgYcHGerjTzNo+Zmbu7BBStBUEzi+7+RBrMrkq/XxeWSBwu"
    "6HyJtgVMl/P08oI1HSWYxYrOkEhAsaLkUz46s/SJtCXmLpdqRjz1Z9Q/loAjCxeSbFUSyUd4nJAeJzYrO4QEPJowGYCZVWkq"
    "qP79acA5wCuqqjovy7IzgNnGKO0wcAD4F7APeBx40swWlJyXgIOK50k83P1lZVm+Mc/zK4HLgPOAMwGyLBvkIReAJ6qq+kcI"
    "YTfwazPb5e6mZlkCvqCprcUriuItIYTrgSvyPD/jKH27eJyRm6XoeGEI4UJgE/AG4EpUJSQB+wtrgcrdN8YYbwghbG7cpJ7b"
    "bvYHbcCMQC1rCYQY4wEpJAGXWlJwGnAL8IEQAo0IFxoHI+awrC6wDCHEo1T/imkT0N1zMyu73e4G4NvABQ3xnq/IbZlKCknA"
    "5+Vz963A99Iotl5OkJ3AqSUJOO0CNuS7mt7qK0tiLMdrK6XQeIQJly8k+a6IMd7R1+QuB4qA0ypgmo1wdz8L+H4IYWYFXpME"
    "nOYImJK/t9JLJpcr8HrUBE+jgCndEt39HcA7l7HPd7QIqDTMFAloqcQ7jzHevMICqAmetlFwGnhURVFszvP8kiRBdgLSLM3j"
    "BR8AFmdExJSlYRwgz/OP0/78az1Nlx8nqmaN24tpEbDe1Mbdzwc2DDGHO2hzmqXHK4EngUfplWPNpb+dCqwGTo0xvjqEUEih"
    "6YqAIUWdzfQqU9pqfuvc4V5gB7AT2Gtmh4/xYTi9ESXVF5wSAesm960tbhlXFynctn///hvWrVs315drbJbr1yX50cye7ksH"
    "iZNZwFT0WaXR7/pU5RLaaHZjjNuzLNtWT+3Vg4xjldunmkPJN0URsC76PD+EcH4L6Ze6+d6VZdm2WjwzK4dIgospygPWsp1N"
    "b+uvOKaAddHCjc3V+1JCAh6Pl7fQ/6vSa99tZvfWuUXpIAEHiYBntSBgfd/7Gks1hQQciDUtyvxX9eUk4EoIWDOnbwCQgIyw"
    "93BbEfCgFJCAvAgS20ICCpSInjqs3iGrntlYriT0MP/vZE6KT7uAMW1HHJf7TdToWwICdNIGRh13r4YYwBTjCJSiX95Wdfig"
    "04cSsB1e0lbfN8b4rRDCwSH6wp7e9C3Aw419aIaqZ6Q3m/NzeiklH3FKsV4Hcy/wof5voZSAJ47Wyu9DCGePeNfV40Ze4JKW"
    "Xss+NcFMbOpk2B1SvbEmZNz/exhYO2YEzJjwbzXItWP7UG++t7gCr96la9THdBa3l0N5QCEkoJCAQkhAIQGFkIBCAgqB8oAc"
    "Y1H6KIloFRJMsYD2ImgBMqkzvQIWLU3BGfAIixsP+RCzDwdUUT29Ah5sUcCPmtkfxqjp00J29QEZtx6w/uajSsWkEnC58VQR"
    "rS+hRmkYIQGFkIBCAgohAYUEFEICCgkohAQUElAICSgkoBASUEhAISSgkIBCSEAhAYWQgEICCiEBhQQUQgIKCSiEBBQSUAgJ"
    "KCSgEBJQSEAhAYWQgEICCiEBhQQUQgIKCSiEBBQSUAgJKCSgEBJQSEAhJKCQgEJIQCEBhZCAQgIKIQHFpJFP4HP2dMR0jELU"
    "Wy8Bx3nOBsy0EPnVAkjAoXkW+DfQHeP5RyADjkgBCTgQZlamH28BvpIksjGa8QAcTI9dSQUJOKiI88C83joJuDIjEHdrUWaX"
    "AhJQ0qA8oFhGFhYWVgGdlh6uqD+bElAcN4ADzM7OrmExjTTOQIpJ7w9LwJXpw56TrrYx+j6iCCiG6MKaxxg39EWxkYkxPqVB"
    "iBg08rm7zwBb2goAIYR9aoLFIPJ1zKyqqup64ILU/IZx37uyLPe2FU0VASdTrNDof/lSg440i9N198uBL6RZnDBmQUYAunme"
    "75GA09uh80EGEu6+Gng/8EVgTZLFxhTQgMcBRcApjHyZmVXuvhXYCjwM7APmyrJ8zt3nO53OqcCZwOuBTcBFffKMNfYALMZ4"
    "X5ZlRf18JOCU5fOAi4F3p6N3QvP8mNK0lC4xwEIIv5zkFIwEHJ95oExHXadY9wmbBbOhxQGf0yslewb4De3lEyXghEbC/Cjn"
    "sk3pmlTpf/3YzJ6Z5OZXaZjJfc/Kbrf7NRUjiOWmBEKM8c7Z2dmHJj36ScDJos4d7g8h3FjPrCgCiuWinjnZZmZ7gWBmUQKK"
    "5aCgVz/4dTP7prvnJ8s6Fgk4OfL9CviYu4dJTrtIwMnq81VAJ8b4E3oVNA4n17IECfjiFK9M700GfCmEsMXMjiT5otaECE7A"
    "diNVY6CRA3uAq8zs06mO0E7GBVmaCWHsmZA6YtE319t/6Uvsb+ONGZMs/e1R4BvAV83soLtnQDxZVwNKwPHoJnlWDVHAwBLF"
    "A88A9wF3AzvN7FCz6gatCxZ9VCkRfDvwYFVVl2VZdjHwKnolWGuBU+jV/q1OazdiCKEL/Bd4Osb4zxDC7rIsd+d5/qCZ/adZ"
    "7pWiXqWF6eJYi+MPALvS0awXXJskXJMuLYRQzM/PF6tWrZozs7klagyfr6KZpr1qJGB7JfkOuJnF1IQeOs59676fJ+mituYQ"
    "rZTkN/avWapQ1M3Mk3BRe8OIE9lEax8blAcUElCIowvYnyid0WkRJ2gRV/+uYEUA9jey8gCvSZ1o03kTbQmYRv2vTBLWM0dz"
    "Abi/bwrpmtSJtjZ3IxVTTWZmMcZ4DYvFtR5jfMTc/SLgoWRmvXv8h81sh7t30u9cIzoxytrlVLlddLvdN3c6nd81muAAvKvO"
    "W33Xe3TdvXL3BXd/n86hoJ2E/UZ3fyo5VqTLB9w9r5vZC4C/0Js6io3KjLuA7wB/Q9+pIYajUxTFuZ1O51rgIyny1cn3HHib"
    "mf3e3D2YWUz7nNxNY/kfi2maI42OoxAD9ftSQGMJpz5jZje7e2Z9m+28F7itsYNTwWKBpBCMWN2dN4LZTWb22briJ9TfFJQk"
    "vAN4E/CLGCP0coKSTzBGnnkmXf4JeHstn5lVZvb/aZZmAaS7b4wxXgdcHkI4W+dSjDAKfgp4ALjbzH66VJHt/wCnKky3OtPf"
    "pwAAAABJRU5ErkJggg=="
)

LETTERTYPES = {
    "Sora": "https://raw.githubusercontent.com/google/fonts/main/ofl/sora/Sora%5Bwght%5D.ttf",
    "Mono": "https://raw.githubusercontent.com/google/fonts/main/ofl/jetbrainsmono/JetBrainsMono%5Bwght%5D.ttf",
}


def haal_lettertypes(map_pad=".", log=print):
    """Probeert de twee lettertypes op te halen. Geeft {naam: pad} van wat lukte."""
    gelukt = {}
    for naam, adres in LETTERTYPES.items():
        doel = Path(map_pad) / f"{naam}.ttf"
        if doel.exists() and doel.stat().st_size > 10000:
            gelukt[naam] = str(doel)
            continue
        try:
            antwoord = requests.get(adres, timeout=30)
            antwoord.raise_for_status()
            doel.write_bytes(antwoord.content)
            gelukt[naam] = str(doel)
        except Exception as fout:
            log(f"  Lettertype {naam} niet opgehaald ({fout}). Het rapport gebruikt een standaardletter.")
    return gelukt


class Rapport(FPDF):
    def __init__(self, lettertypes):
        super().__init__(orientation="P", unit="mm", format="A4")
        self.set_auto_page_break(False)
        self.heeft_sora = "Sora" in lettertypes
        self.heeft_mono = "Mono" in lettertypes
        if self.heeft_sora:
            self.add_font("Sora", "", lettertypes["Sora"])
        if self.heeft_mono:
            self.add_font("Mono", "", lettertypes["Mono"])

    def kop_font(self, grootte):
        self.set_font("Sora" if self.heeft_sora else "Helvetica", "" if self.heeft_sora else "B", grootte)

    def tekst_font(self, grootte):
        self.set_font("Sora" if self.heeft_sora else "Helvetica", "", grootte)

    def mono_font(self, grootte):
        self.set_font("Mono" if self.heeft_mono else "Courier", "", grootte)

    def nieuwe_pagina(self):
        self.add_page()
        self.set_fill_color(*ACHTERGROND)
        self.rect(0, 0, 210, 297, style="F")
        self.logo()

    def logo(self, x=182, y=13, breedte=12):
        """Het logo rechtsboven. Stil falen als het beeld niet leesbaar is; een
        rapport zonder logo is bruikbaar, een rapport dat niet opent niet."""
        try:
            self.image(io.BytesIO(base64.b64decode(LOGO_B64)), x=x, y=y, w=breedte)
        except Exception:
            pass

    def label(self, tekst, x, y):
        self.mono_font(7)
        self.set_text_color(*SUBTIEL)
        self.set_xy(x, y)
        self.cell(0, 4, tekst.upper())

    def kaart(self, x, y, breedte, hoogte):
        self.set_fill_color(*KAART)
        self.set_draw_color(*RAND)
        self.set_line_width(0.2)
        self.rect(x, y, breedte, hoogte, style="DF")


def _kort(tekst, maximum):
    tekst = str(tekst)
    return tekst if len(tekst) <= maximum else tekst[: maximum - 1] + "…"


def grafiek_onderdelen(rijen, aantal_zoektermen):
    """Staafdiagram van de onderdelen, in huisstijl, als PNG in het geheugen."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    top = [r for r in rijen if r["onderdeel"] != "organic"][:10]
    if not top:
        return None
    namen = [_kort(nl(r["onderdeel"]), 28) for r in top][::-1]
    waarden = [r["aandeel"] * 100 for r in top][::-1]

    fig, as_ = plt.subplots(figsize=(7.2, max(2.2, 0.42 * len(top))), dpi=200)
    fig.patch.set_facecolor("#16272F")
    as_.set_facecolor("#16272F")
    as_.barh(namen, waarden, color="#FF5F1F", height=0.62)
    as_.set_xlim(0, 100)
    as_.set_xlabel("% van de zoektermen", color="#8A9BA3", fontsize=8)
    for kant in ("top", "right", "left"):
        as_.spines[kant].set_visible(False)
    as_.spines["bottom"].set_color("#1F343E")
    as_.tick_params(colors="#8A9BA3", labelsize=8, length=0)
    as_.grid(axis="x", color="#1F343E", linewidth=0.6)
    as_.set_axisbelow(True)
    for i, waarde in enumerate(waarden):
        as_.text(waarde + 1.5, i, f"{waarde:.0f}%", va="center", color="#F4F6F7", fontsize=8)
    fig.tight_layout()

    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", facecolor=fig.get_facecolor())
    plt.close(fig)
    buffer.seek(0)
    return buffer

def maak_rapport(pad, *, clusters, families, onderdeel_rijen, concurrent_rijen,
                 zoektermen, kosten, datum, locatie_naam, eigen_domein="",
                 lettertypes=None, log=print):
    lettertypes = lettertypes if lettertypes is not None else haal_lettertypes(log=log)
    pdf = Rapport(lettertypes)

    # ---------------- pagina 1 ----------------
    pdf.nieuwe_pagina()

    pdf.label("Zoektermanalyse", 16, 16)
    pdf.kop_font(19)
    pdf.set_text_color(*TEKST)
    pdf.set_xy(16, 22)
    pdf.cell(160, 10, "Zoektermen gegroepeerd op wat Google toont")

    pdf.tekst_font(9)
    pdf.set_text_color(*GEDEMPT)
    pdf.set_xy(16, 33)
    pdf.cell(0, 5, f"{len(zoektermen)} zoektermen  ·  {locatie_naam}  ·  gemeten op {datum}")

    pdf.set_draw_color(*RAND)
    pdf.line(16, 41, 194, 41)

    # Tegels
    tegels = [
        (str(len(zoektermen)), "zoektermen"),
        (str(len(clusters)), "onderwerpen"),
        (str(len(families)), "families"),
        (f"${kosten:.2f}", "meetkosten"),
    ]
    breedte, x = 42, 16
    for waarde, omschrijving in tegels:
        pdf.kaart(x, 47, breedte, 22)
        pdf.kop_font(16)
        pdf.set_text_color(*TEKST)
        pdf.set_xy(x + 6, 51)
        pdf.cell(0, 8, waarde)
        pdf.mono_font(7)
        pdf.set_text_color(*SUBTIEL)
        pdf.set_xy(x + 6, 60)
        pdf.cell(0, 4, omschrijving.upper())
        x += breedte + 3.5

    # Clusters
    pdf.label("De onderwerpen", 16, 78)
    pdf.tekst_font(8.5)
    pdf.set_text_color(*GEDEMPT)
    pdf.set_xy(16, 83)
    pdf.multi_cell(178, 4.4,
                   "Zoektermen in hetzelfde onderwerp delen genoeg zoekresultaten om als één pagina "
                   "behandeld te worden. Onderwerpen met hetzelfde familienummer horen bij elkaar "
                   "maar verdienen een eigen pagina.")

    y = 96
    pdf.mono_font(7)
    pdf.set_text_color(*SUBTIEL)
    pdf.set_xy(16, y)
    pdf.cell(14, 5, "FAMILIE")
    pdf.set_xy(32, y)
    pdf.cell(14, 5, "AANTAL")
    pdf.set_xy(50, y)
    pdf.cell(140, 5, "ZOEKTERMEN")
    y += 6
    pdf.set_draw_color(*RAND)
    pdf.line(16, y - 1, 194, y - 1)

    for cluster in clusters:
        if y > 268:
            pdf.tekst_font(8)
            pdf.set_text_color(*SUBTIEL)
            pdf.set_xy(16, y + 2)
            pdf.cell(0, 4, "De volledige lijst staat in het CSV-bestand.")
            break
        regel = ", ".join(cluster["termen"])
        regels = pdf.multi_cell(140, 4.6, _kort(regel, 190), dry_run=True, output="LINES")
        hoogte = max(6.0, 4.6 * len(regels))

        pdf.mono_font(8)
        pdf.set_text_color(*ACCENT if cluster["omvang"] > 1 else SUBTIEL)
        pdf.set_xy(16, y)
        pdf.cell(14, 4.6, f"{cluster['familie'] + 1:02d}")
        pdf.set_text_color(*GEDEMPT)
        pdf.set_xy(32, y)
        pdf.cell(14, 4.6, str(cluster["omvang"]))

        pdf.tekst_font(8.5)
        pdf.set_text_color(*TEKST)
        pdf.set_xy(50, y)
        pdf.multi_cell(140, 4.6, _kort(regel, 190))
        y += hoogte + 1.6

    pdf.mono_font(7)
    pdf.set_text_color(*SUBTIEL)
    pdf.set_xy(16, 284)
    pdf.cell(0, 4, "GEMAAKT MET HET ZOEKTERMSCRIPT VAN FERRYJANSEN.NL/TOOLS")

    # ---------------- pagina 2 ----------------
    pdf.nieuwe_pagina()

    pdf.label("Zoektermanalyse", 16, 16)
    pdf.kop_font(18)
    pdf.set_text_color(*TEKST)
    pdf.set_xy(16, 22)
    pdf.cell(160, 9, "Wat Google in deze zoekresultaten toont")

    pdf.tekst_font(8.5)
    pdf.set_text_color(*GEDEMPT)
    pdf.set_xy(16, 33)
    pdf.multi_cell(178, 4.4,
                   "Bij welk deel van je zoektermen verschijnt welk onderdeel. Dit zegt iets over "
                   "wat Google bij deze vraag als antwoord beschouwt, en over hoeveel ruimte er "
                   "voor een gewoon resultaat overblijft.")

    plaatje = grafiek_onderdelen(onderdeel_rijen, len(zoektermen))
    if plaatje is not None:
        pdf.image(plaatje, x=16, y=44, w=178)

    y = 44 + (min(10, max(1, len([r for r in onderdeel_rijen if r["onderdeel"] != "organic"]))) * 10.6) + 14
    y = min(y, 150)

    pdf.label("Wie er het vaakst staat", 16, y)
    y += 5
    pdf.tekst_font(8.5)
    pdf.set_text_color(*GEDEMPT)
    pdf.set_xy(16, y)
    pdf.multi_cell(178, 4.4,
                   "Gewogen naar positie, dus een eerste plek telt zwaarder dan een tiende. "
                   "Dit is wie er in deze verzameling zoektermen de meeste ruimte inneemt.")
    y += 11

    pdf.mono_font(7)
    pdf.set_text_color(*SUBTIEL)
    pdf.set_xy(16, y)
    pdf.cell(90, 5, "DOMEIN")
    pdf.set_xy(112, y)
    pdf.cell(30, 5, "ZOEKTERMEN")
    pdf.set_xy(150, y)
    pdf.cell(30, 5, "ZICHTBAARHEID")
    y += 6
    pdf.line(16, y - 1, 194, y - 1)

    for rij in concurrent_rijen[:16]:
        if y > 272:
            break
        eigen = rij["is_eigen"]
        pdf.tekst_font(8.5)
        pdf.set_text_color(*(ACCENT if eigen else TEKST))
        pdf.set_xy(16, y)
        pdf.cell(90, 4.6, _kort(rij["domein"], 46) + ("  (jij)" if eigen else ""))
        pdf.mono_font(8)
        pdf.set_text_color(*GEDEMPT)
        pdf.set_xy(112, y)
        pdf.cell(30, 4.6, f"{rij['zoektermen']}  ({rij['aandeel'] * 100:.0f}%)")
        pdf.set_xy(150, y)
        pdf.cell(30, 4.6, f"{rij['zichtbaarheid']:.2f}")
        y += 5.4

    pdf.mono_font(7)
    pdf.set_text_color(*SUBTIEL)
    pdf.set_xy(16, 284)
    pdf.cell(0, 4, "GEMAAKT MET HET ZOEKTERMSCRIPT VAN FERRYJANSEN.NL/TOOLS")

    pdf.output(pad)
    return pad

In [ ]:
import csv

datum = datetime.date.today().strftime("%d-%m-%Y")

# ── CSV-bestanden ──────────────────────────────────────────────────────────────
def schrijf_csv(naam, kolommen, rijen):
    with open(naam, "w", newline="", encoding="utf-8-sig") as f:
        schrijver = csv.writer(f, delimiter=";")
        schrijver.writerow(kolommen)
        schrijver.writerows(rijen)
    return naam

bestanden = [
    # Het werkblad eerst: dit is het bestand waar je mee verder werkt.
    maak_werkblad(
        "zoektermanalyse.xlsx",
        clusters=clusters, onderdeel_rijen=onderdeel_rijen,
        concurrent_rijen=concurrent_rijen, paren=paren,
        zoektermen=list(serps), top_n=TOP_N, samen_vanaf=SAMEN_VANAF,
        verwant_vanaf=VERWANT_VANAF, datum=datum, locatie_naam=LOCATIE_NAAM,
    ),
    schrijf_csv("onderwerpen.csv", ["familie", "aantal", "zoektermen"],
                [[c["familie"] + 1, c["omvang"], ", ".join(c["termen"])] for c in clusters]),
    schrijf_csv("onderdelen.csv", ["onderdeel", "zoektermen", "aandeel", "prominentie"],
                [[nl(r["onderdeel"]), r["zoektermen"], r["aandeel"], r["prominentie"]]
                 for r in onderdeel_rijen]),
    schrijf_csv("domeinen.csv", ["domein", "zoektermen", "aandeel", "zichtbaarheid"],
                [[r["domein"], r["zoektermen"], r["aandeel"], r["zichtbaarheid"]]
                 for r in concurrent_rijen]),
]

# ── Het rapport ────────────────────────────────────────────────────────────────
pdf_pad = maak_rapport(
    "zoektermanalyse.pdf",
    clusters=clusters, families=families,
    onderdeel_rijen=onderdeel_rijen, concurrent_rijen=concurrent_rijen,
    zoektermen=list(serps), kosten=kosten, datum=datum,
    locatie_naam=LOCATIE_NAAM, eigen_domein=EIGEN_DOMEIN,
)
bestanden.append(pdf_pad)

print("Klaar. Deze bestanden staan er nu:")
for b in bestanden:
    print(f"  {b}")

try:
    from google.colab import files
    for b in bestanden:
        files.download(b)
except ImportError:
    print("\n(Je draait dit buiten Colab. De bestanden staan in deze map.)")

---

## Wat je met de uitkomst doet

**De onderwerpen.** Elke groep met meer dan één zoekterm is één pagina. Zoektermen met
hetzelfde familienummer maar in een andere groep horen bij elkaar en verdienen een eigen
pagina, met een link naar elkaar.

**De losse zoektermen.** Een zoekterm die in zijn eentje overblijft is niet onbelangrijk.
Hij deelt alleen weinig met de rest van je lijst. Dat kan betekenen dat het een eigen
onderwerp is, of dat er nog zoektermen in je lijst ontbreken die de brug vormen.

**Wat Google toont.** Staan er bij het merendeel van je zoektermen andere vragen of een
AI-overzicht, dan is de eerste plek minder waard dan je denkt en is het de moeite waard
om te kijken welke vraag daar beantwoord wordt.

**De domeinen.** Wie er in jouw verzameling het vaakst staat, is je werkelijke
concurrent in de zoekresultaten. Dat is vaak niet dezelfde partij als je commerciële
concurrent.

---

Dit notebook hoort bij een uitleg op [ferryjansen.nl/tools](https://ferryjansen.nl/tools/).
Vragen of iets kapot? Laat het weten.